# Representasi TF-IDF dan Reduksi Dimensi Data Berita

Setelah data berita berhasil dikumpulkan dan melalui tahap preprocessing, data teks perlu diubah menjadi bentuk numerik agar dapat diolah lebih lanjut. Pada tahap ini, teks berita direpresentasikan menggunakan TF-IDF (Term Frequency–Inverse Document Frequency) untuk memberikan bobot pada setiap kata berdasarkan tingkat kepentingannya dalam dokumen.

Data hasil TF-IDF memiliki jumlah fitur yang cukup banyak karena setiap kata yang terdapat dalam vocabulary menjadi sebuah fitur. Oleh karena itu, dilakukan reduksi dimensi menggunakan PCA (Principal Component Analysis) untuk mengurangi jumlah fitur dengan tetap mempertahankan sebagian besar informasi dari data.

Tahapan yang dilakukan meliputi pembentukan vocabulary, representasi teks menggunakan TF-IDF, dan reduksi dimensi menggunakan PCA.

### Install Library NLTK

In [2]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('words')

[nltk_data] Downloading package punkt to C:\Users\triad/nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\triad/nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package words to C:\Users\triad/nltk_data...
[nltk_data]   Unzipping corpora\words.zip.


True

In [37]:
import pandas as pd

df = pd.read_csv("dataset_berita.csv")
print(df.shape)
df.head()

(200, 3)


,id,isi_berita,label
0,1,Persib Bandung mengawali kiprahnya di AFC Cham...,sport
1,2,"Alwi Farhan, Moh Zaki Ubaidillah dan Muhamad Y...",sport
2,3,Tottenham Hotspur akhirnya kembali mencetak go...,sport
3,4,Yan Diomande tampil menawan saat Real Madrid m...,sport
4,5,"Pelatih Timnas Jerman, Juergen Klopp kabarnya ...",sport


## Preprocessing Data

Data berita yang telah dikumpulkan terlebih dahulu dilakukan preprocessing. Pada tahap cleaning, teks diubah menjadi huruf kecil, URL, emoji, tanda baca, dan angka dihapus, serta spasi yang berlebih dirapikan. Setelah itu, teks diubah menjadi token atau kata-kata menggunakan proses tokenisasi.

In [38]:
import re
import emoji

def cleaning(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['isi_bersih'] = df['isi_berita'].apply(cleaning)
df[['isi_berita', 'isi_bersih']].head()

,isi_berita,isi_bersih
0,Persib Bandung mengawali kiprahnya di AFC Cham...,persib bandung mengawali kiprahnya di afc cham...
1,"Alwi Farhan, Moh Zaki Ubaidillah dan Muhamad Y...",alwi farhan moh zaki ubaidillah dan muhamad yu...
2,Tottenham Hotspur akhirnya kembali mencetak go...,tottenham hotspur akhirnya kembali mencetak go...
3,Yan Diomande tampil menawan saat Real Madrid m...,yan diomande tampil menawan saat real madrid m...
4,"Pelatih Timnas Jerman, Juergen Klopp kabarnya ...",pelatih timnas jerman juergen klopp kabarnya m...


In [39]:
from nltk.tokenize import word_tokenize

df['tokens'] = df['isi_bersih'].apply(word_tokenize)
df[['isi_bersih', 'tokens']].head()

,isi_bersih,tokens
0,persib bandung mengawali kiprahnya di afc cham...,"[persib, bandung, mengawali, kiprahnya, di, af..."
1,alwi farhan moh zaki ubaidillah dan muhamad yu...,"[alwi, farhan, moh, zaki, ubaidillah, dan, muh..."
2,tottenham hotspur akhirnya kembali mencetak go...,"[tottenham, hotspur, akhirnya, kembali, mencet..."
3,yan diomande tampil menawan saat real madrid m...,"[yan, diomande, tampil, menawan, saat, real, m..."
4,pelatih timnas jerman juergen klopp kabarnya m...,"[pelatih, timnas, jerman, juergen, klopp, kaba..."


Selanjutnya dilakukan stopword removal untuk menghapus kata-kata yang dianggap kurang memiliki informasi penting. Stopword yang digunakan berasal dari Sastrawi dan ditambahkan beberapa kata secara manual.

In [41]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stopword_factory = StopWordRemoverFactory()
indo_stopwords = set(stopword_factory.get_stop_words())

tambahan_stopwords = {"kali", "mana", "hal", "tim", "era", "pun", "apa", "dan", "dari", "akan", "yang", "saya"}
indo_stopwords = indo_stopwords.union(tambahan_stopwords)

def kata_asing(tokens):
    asing = []
    for t in tokens:
        if t in indo_stopwords:
            continue
        if len(t) <= 3:
            continue
        stem = stemmer.stem(t)
        if t in english_vocab and stem == t:
            asing.append(t)
    return asing

contoh = df['tokens'].iloc[0]
print(kata_asing(contoh))

['league', 'world', 'stadium', 'tuan', 'julio', 'marc', 'bola', 'kang', 'tuan', 'dong', 'ragnar', 'tuan', 'super', 'league', 'momentum', 'masa', 'injury', 'time', 'league', 'kang', 'arab', 'shin', 'moon', 'park', 'woon', 'gabriel', 'julio', 'ikhwan', 'patricio', 'adam', 'marc', 'ragnar', 'balsa']


In [42]:
def remove_stopwords(tokens):
    return [t for t in tokens if t not in indo_stopwords]

df['tokens_no_stopword'] = df['tokens'].apply(remove_stopwords)
df[['tokens', 'tokens_no_stopword']].head()

,tokens,tokens_no_stopword
0,"[persib, bandung, mengawali, kiprahnya, di, af...","[persib, bandung, mengawali, kiprahnya, afc, c..."
1,"[alwi, farhan, moh, zaki, ubaidillah, dan, muh...","[alwi, farhan, moh, zaki, ubaidillah, muhamad,..."
2,"[tottenham, hotspur, akhirnya, kembali, mencet...","[tottenham, hotspur, akhirnya, mencetak, gol, ..."
3,"[yan, diomande, tampil, menawan, saat, real, m...","[yan, diomande, tampil, menawan, real, madrid,..."
4,"[pelatih, timnas, jerman, juergen, klopp, kaba...","[pelatih, timnas, jerman, juergen, klopp, kaba..."


Setelah stopword dihapus, dilakukan stemming menggunakan stemmer Bahasa Indonesia untuk mengubah kata menjadi bentuk dasarnya. Dari proses ini dihasilkan kolom tokens_final yang digunakan pada tahap berikutnya.

In [43]:
def stem_tokens(tokens):
    return [stemmer.stem(t) for t in tokens]

hasil_stem = []
total = len(df)

for i, tokens in enumerate(df['tokens_no_stopword'], 1):
    hasil_stem.append(stem_tokens(tokens))
    print(f"\rStemming progress: {i}/{total}", end="", flush=True)

df['tokens_final'] = hasil_stem

print("\nSelesai stemming")
df[['tokens_no_stopword', 'tokens_final']].head()

Stemming progress: 200/200
Selesai stemming


,tokens_no_stopword,tokens_final
0,"[persib, bandung, mengawali, kiprahnya, afc, c...","[persib, bandung, awal, kiprah, afc, champions..."
1,"[alwi, farhan, moh, zaki, ubaidillah, muhamad,...","[alwi, farhan, moh, zaki, ubaidillah, muhamad,..."
2,"[tottenham, hotspur, akhirnya, mencetak, gol, ...","[tottenham, hotspur, akhir, cetak, gol, lawan,..."
3,"[yan, diomande, tampil, menawan, real, madrid,...","[yan, diomande, tampil, tawan, real, madrid, m..."
4,"[pelatih, timnas, jerman, juergen, klopp, kaba...","[latih, timnas, jerman, juergen, klopp, kabar,..."


## Pembentukan Vocabulary

Setelah preprocessing selesai, seluruh kata dari tokens_final dikumpulkan dan kata yang sama hanya dihitung satu kali. Hasilnya terdapat 5.211 kata unik yang digunakan sebagai vocabulary.

In [44]:
semua_kata = [kata for tokens in df['tokens_final'] for kata in tokens]
reserved_words = {"id", "label"}

vocabulary = sorted(set(
    kata for kata in semua_kata 
    if kata not in reserved_words
))
print(f"Jumlah kata unik setelah filter: {len(vocabulary)}")
print(vocabulary[:30])

Jumlah kata unik setelah filter: 5211
['a', 'aaa', 'ab', 'abad', 'abadi', 'abai', 'abal', 'abang', 'abdoul', 'abdulelah', 'abdullah', 'abel', 'abet', 'abha', 'abiel', 'absen', 'absorber', 'ac', 'academy', 'acara', 'accelerating', 'acceptance', 'access', 'accessories', 'accident', 'accreditation', 'acd', 'aceh', 'achadie', 'achilles']


## Representasi Berita dengan TF-IDF

TF-IDF digunakan untuk memberikan bobot pada setiap kata berdasarkan seberapa penting kata tersebut dalam suatu berita. Pada proses ini terdapat dua perhitungan utama, yaitu TF (Term Frequency) dan IDF (Inverse Document Frequency).

Term Frequency (TF):
$$
TF(t,d) = \frac{f(t,d)}{\sum_{t' \in d} f(t',d)}
$$

Keterangan:
* $t$ = kata yang dihitung
* $d$ = dokumen/berita
* $f(t,d)$ = jumlah kemunculan kata $t$ dalam dokumen $d$
* Penyebut = jumlah seluruh kata dalam dokumen

Inverse Document Frequency (IDF):
$$
IDF(t) = \log\left(\frac{N}{df(t)}\right)
$$

Keterangan:
* $N$ = jumlah seluruh dokumen
* $df(t)$ = jumlah dokumen yang mengandung kata t

Kemudian nilai TF dan IDF dikalikan untuk mendapatkan bobot TF-IDF:
$$TFIDF(t,d)=TF(t,d)×IDF(t)$$

In [45]:
from sklearn.feature_extraction.text import CountVectorizer

df['teks_final'] = df['tokens_final'].apply(lambda tokens: ' '.join(tokens))

vectorizer = CountVectorizer(binary=True, vocabulary=vocabulary)
onehot_matrix = vectorizer.fit_transform(df['teks_final'])

onehot_df = pd.DataFrame(onehot_matrix.toarray(), columns=vectorizer.get_feature_names_out())
onehot_df.insert(0, 'id', df['id'].values)
onehot_df['label'] = df['label'].values

print(onehot_df.shape)
onehot_df.head()

(200, 5213)


,id,a,aaa,ab,abad,abadi,abai,abal,abang,abdoul,...,zerbi,zhegrova,zhejiang,zhestkova,zielinski,zikrak,zim,zona,zubimendi,label
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,sport
1,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,sport
2,3,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,sport
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,sport
4,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,sport


In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(vocabulary=vocabulary)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['teks_final'])

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(), 
    columns=tfidf_vectorizer.get_feature_names_out()
)

tfidf_df.insert(0, 'id', df['id'].values)
tfidf_df['label'] = df['label'].values

print(tfidf_df.shape)
tfidf_df.head()

(200, 5213)


,id,a,aaa,ab,abad,abadi,abai,abal,abang,abdoul,...,zerbi,zhegrova,zhejiang,zhestkova,zielinski,zikrak,zim,zona,zubimendi,label
0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sport
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sport
2,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.133095,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sport
3,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sport
4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sport


Teks yang telah melalui preprocessing kemudian digabungkan kembali menjadi teks menggunakan tokens_final. Selanjutnya, TF-IDF digunakan untuk mengubah setiap berita menjadi representasi numerik berdasarkan bobot kepentingan setiap kata dalam dokumen. Hasil representasi menghasilkan 200 berita dengan 5.211 fitur kata.

In [48]:
contoh_doc = tfidf_df.drop(columns=['id', 'label']).iloc[0]
print(contoh_doc.sort_values(ascending=False).head(10))

persib     0.397914
seoul      0.285760
bandung    0.202968
bikin      0.198393
fc         0.176851
cesar      0.170772
jeong      0.170772
ancam      0.142880
min        0.113848
kang       0.113848
Name: 0, dtype: float64


Pada berita pertama, beberapa kata dengan nilai TF-IDF tertinggi adalah persib, seoul, dan bandung. Hal ini menunjukkan bahwa kata-kata tersebut memiliki bobot yang cukup tinggi pada dokumen tersebut dibandingkan kata lainnya.

In [49]:
tfidf_df.to_csv("tfidf_berita.csv", index=False)
print("Selesai! Tersimpan sebagai tfidf_berita.csv")

Selesai! Tersimpan sebagai tfidf_berita.csv


Hasil TF-IDF kemudian disimpan dalam file tfidf_berita.csv untuk digunakan pada proses selanjutnya.

## Reduksi Dimensi dengan PCA

Data hasil TF-IDF kemudian digunakan sebagai input untuk PCA (Principal Component Analysis). Sebelum PCA, data memiliki ukuran 200 × 5.211, yaitu 200 berita dengan 5.211 fitur.

In [50]:
X = tfidf_df.drop(columns=['id', 'label']).values
y = tfidf_df['label'].values

print("Shape sebelum PCA:", X.shape)

Shape sebelum PCA: (200, 5211)


PCA digunakan untuk mengurangi jumlah fitur dengan mempertahankan informasi utama dari data. Pada notebook, jumlah komponen ditentukan berdasarkan cumulative explained variance sebesar 90%. Hasilnya, dibutuhkan 142 komponen dari 5.211 dimensi awal.

In [51]:
from sklearn.decomposition import PCA
import numpy as np

pca_full = PCA()
pca_full.fit(X)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

n_components_90 = np.argmax(cumulative_variance >= 0.90) + 1
print(f"Jumlah komponen untuk menjelaskan 90% variansi: {n_components_90}")
print(f"Total dimensi awal: {X.shape[1]}")

Jumlah komponen untuk menjelaskan 90% variansi: 142
Total dimensi awal: 5211


In [52]:
pca = PCA(n_components=n_components_90)
X_reduced = pca.fit_transform(X)

print("Shape setelah PCA:", X_reduced.shape)

pca_df = pd.DataFrame(X_reduced, columns=[f'PC{i+1}' for i in range(X_reduced.shape[1])])
pca_df.insert(0, 'id', tfidf_df['id'].values)
pca_df['label'] = y

pca_df.head()

Shape setelah PCA: (200, 142)


,id,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,...,PC134,PC135,PC136,PC137,PC138,PC139,PC140,PC141,PC142,label
0,1,-0.178188,-0.028485,0.061914,0.180519,0.150416,0.106157,0.388521,0.365211,-0.163586,...,-0.014949,0.165988,0.147785,0.145939,0.113244,-0.060222,0.041977,0.179442,-0.164346,sport
1,2,-0.084644,-0.004242,-0.119602,0.215207,-0.112751,0.032872,-0.327947,-0.044506,-0.032662,...,0.144254,0.052055,0.084368,-0.037805,0.096895,-0.065028,0.026114,0.005929,0.143086,sport
2,3,-0.283491,-0.088761,0.229946,-0.034866,0.278287,-0.053873,0.061809,-0.223259,-0.220632,...,-0.036292,0.018864,-0.116040,-0.153706,0.044965,0.136476,-0.069549,-0.005644,0.021088,sport
3,4,-0.213099,-0.052570,0.150552,-0.027003,0.351577,-0.118623,-0.079965,0.096851,0.289816,...,0.077403,-0.109959,0.023749,-0.027567,-0.015360,-0.108827,0.076566,0.123479,-0.054462,sport
4,5,-0.069812,-0.017989,-0.024301,0.072706,0.005293,0.011781,-0.012406,0.031740,0.052686,...,-0.004005,-0.022047,-0.012791,0.018577,-0.023309,-0.004288,-0.021912,0.016455,-0.033723,sport


Setelah dilakukan PCA, ukuran data berubah menjadi 200 × 142. Setiap fitur hasil reduksi direpresentasikan sebagai PC1, PC2, hingga PC142.

In [33]:
pca_df.to_csv("pca_berita.csv", index=False)
print("Selesai! Tersimpan sebagai pca_berita.csv")

Selesai! Tersimpan sebagai pca_berita.csv


Hasil akhir PCA kemudian disimpan dalam file pca_berita.csv.